[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Red1-Rahman/NiriZan/blob/main/experiments/08a_external_validation_and_scale-GroqAPI.ipynb)
[![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/kernels/welcome?src=https://github.com/Red1-Rahman/NiriZan/blob/main/experiments/08a_external_validation_and_scale-GroqAPI.ipynb)

# 08: External Validation and Scale

**Goal:** externally validate NiriZan's RAG Triad against **RAGAS, DeepEval, and Opik** on three public RAG benchmarks using one reproducible Groq judge configuration.

### Evaluation matrix

| Section | Dataset | Frameworks |
|---|---|---|
| **A** | AmnestyQA | NiriZan, RAGAS, DeepEval, Opik |
| **B** | HotpotQA | NiriZan, RAGAS, DeepEval, Opik |
| **C** | MultiHop-RAG | NiriZan, RAGAS, DeepEval, Opik |

This notebook evaluates the **evaluators**. The application answer is the benchmark-supplied answer, so all four frameworks receive the same question, retrieved context, and answer. No separate answer-generating model is introduced.

The default live budget is deliberately bounded for the Groq free tier:

- AmnestyQA: **20/20**
- HotpotQA: **30**
- MultiHop-RAG: **30**
- Total: **80 items**

Every completed score is cached locally so an interrupted run can resume without repeating successful judge calls.

**GPU:** Colab T4 is recommended because NiriZan's validation path loads a Sentence Transformers scorer. The external judges run remotely through Groq.

## 1. Environment Setup

The notebook installs NiriZan from the current `main` branch, following the established experiment convention, plus only the dependencies needed for real benchmark scoring.

The default Groq judge is **`openai/gpt-oss-20b`**. It is the practical free-tier choice for this experiment: Groq currently lists a larger free daily token allowance for GPT OSS 20B than for Llama 3.3 70B while retaining a large context window. Set `NIRIZAN_JUDGE_MODEL=llama-3.3-70b-versatile` only when a larger judge is worth the additional quota pressure.

Create a Colab Secret named `GROQ_API_KEY`. The notebook never writes the key to disk.

In [1]:
# Cell 1 — Install and configure the evaluation stack
#
# RAGAS 0.4.3 currently has an Instructor structured-output integration
# that is sensitive to the Instructor version. Pin the Instructor version
# so RAGAS does not receive an incompatible patched-client interface.

import os
import sys
import subprocess
import warnings

IS_COLAB = bool(os.environ.get("COLAB_RELEASE_TAG"))

packages = [
    "pydantic>=2.7,<3",
    "numpy",
    "scipy",
    "pandas",
    "matplotlib",
    "datasets",
    "sentence-transformers",

    # Evaluation frameworks
    "ragas==0.4.3",
    "instructor==1.14.3",
    "deepeval>=4,<5",
    "opik>=1,<2",

    # Groq / OpenAI-compatible API
    "groq",
    "litellm",
    "openai==2.45.0",

    # LangChain compatibility
    "langchain-community",
    "langchain-openai",

    # NiriZan
    "git+https://github.com/Red1-Rahman/NiriZan.git@main#egg=nirizan",
]

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        *packages,
    ]
)

if IS_COLAB:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "requests==2.32.4",
        ]
    )

warnings.filterwarnings(
    "ignore",
    category=DeprecationWarning,
)
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)


# ---------------------------------------------------------------------
# Secrets
# ---------------------------------------------------------------------

def get_secret(name: str) -> str | None:
    value = os.environ.get(name)

    if value:
        return value

    if IS_COLAB:
        try:
            from google.colab import userdata

            value = userdata.get(name)
            return str(value) if value else None
        except Exception:
            return None

    return None


GROQ_API_KEY = get_secret("GROQ_API_KEY")

if GROQ_API_KEY:
    os.environ["GROQ_API_KEY"] = GROQ_API_KEY

    # OpenAI-compatible clients can use the Groq endpoint.
    os.environ["OPENAI_API_KEY"] = GROQ_API_KEY
    os.environ["OPENAI_BASE_URL"] = (
        "https://api.groq.com/openai/v1"
    )


# ---------------------------------------------------------------------
# Judge configuration
# ---------------------------------------------------------------------

JUDGE_MODEL = os.environ.get(
    "NIRIZAN_JUDGE_MODEL",
    "openai/gpt-oss-20b",
)

GROQ_MODEL = JUDGE_MODEL.removeprefix("openai/")

JUDGE_BASE_URL = (
    "https://api.groq.com/openai/v1"
)

os.environ["OPIK_TRACK_DISABLE"] = "true"


# ---------------------------------------------------------------------
# Runtime information
# ---------------------------------------------------------------------

print(
    "runtime:",
    "colab" if IS_COLAB else "local",
)

print(
    "Groq API key present:",
    bool(GROQ_API_KEY),
)

print(
    "judge model:",
    GROQ_MODEL,
)

print(
    "judge endpoint:",
    JUDGE_BASE_URL,
)

print()
print("IMPORTANT: restart the Colab runtime now.")
print(
    "After restarting, rerun Cell 1 and continue from Cell 2."
)

runtime: colab
Groq API key present: True
judge model: gpt-oss-20b
judge endpoint: https://api.groq.com/openai/v1

IMPORTANT: restart the Colab runtime now.
After restarting, rerun Cell 1 and continue from Cell 2.


In [2]:
import json
import sqlite3
import time
import hashlib
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats as scipy_stats
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, util

from nirizan import __version__ as NIRIZAN_VERSION
from nirizan.instrumentation.spans import SpanKind, Trace
from nirizan.instrumentation.tracer import Tracer
from nirizan.orchestrator.collector import TraceCollector, CollectorExporter
from nirizan.metrics.rag_triad import RAGTriadMetric
from nirizan.regression.comparator import BaselineComparator, RegressionSeverity

RNG_SEED = 42
LIMITS = {"amnesty_qa": 20, "hotpot_qa": 30, "multihop_rag": 30}

RESULTS_DIR = Path("external_validation_results")
RESULTS_DIR.mkdir(exist_ok=True)
CACHE_PATH = RESULTS_DIR / "scores.sqlite"

print("NiriZan version:", NIRIZAN_VERSION)
print("Evaluation limits:", LIMITS)

NiriZan version: 0.1.0
Evaluation limits: {'amnesty_qa': 20, 'hotpot_qa': 30, 'multihop_rag': 30}


## 2. Metric Mapping

The comparison is **dimension-aligned**, rather than an arbitrary average of each framework's metrics.

| Quality dimension | NiriZan | RAGAS | DeepEval | Opik |
|---|---|---|---|---|
| Context relevance | `context_relevance` | `ContextPrecisionWithoutReference` | `ContextualRelevancyMetric` | `ContextPrecision` |
| Groundedness | `groundedness` | `Faithfulness` | `FaithfulnessMetric` | `1 - Hallucination` |
| Answer relevance | `answer_relevance` | `AnswerRelevancy` | `AnswerRelevancyMetric` | `AnswerRelevance` |

These are **related metrics, not identical algorithms**. In particular, Opik's `ContextPrecision` is reference-oriented while DeepEval's `ContextualRelevancyMetric` is referenceless. The notebook therefore reports agreement **per dimension** and preserves each framework's native metric identity.

Opik's Hallucination metric has the opposite score direction: `0` means no hallucination and `1` means hallucination detected. It is inverted before comparison so all reported groundedness scores are higher-is-better.

## 3. Public Dataset Acquisition

All benchmark adapters normalize to:

`item_id`, `question`, `contexts`, `answer`, and `reference`.

No synthetic fallback is used. A failed public dataset download stops the notebook rather than silently replacing a benchmark.

### A — AmnestyQA

The English `eval` split contains 20 curated items.

### B — HotpotQA

A deterministic 30-item slice of the `validation` split from the `distractor` configuration is used, preserving its supplied distractor contexts.

### C — MultiHop-RAG

The benchmark provides both a query configuration and a corpus configuration. For each selected query, the required evidence documents are reconstructed from `evidence_list`; three deterministic distractor documents from the same corpus are then added and shuffled with a fixed seed. This keeps the multi-document setting while making retrieval-quality metrics non-trivial.

In [3]:
DATASET_SOURCES = {}

def load_amnesty():
    ds = load_dataset("explodinggradients/amnesty_qa", "english_v3", split="eval")
    items = []
    for i, row in enumerate(ds):
        items.append({
            "item_id": f"amnesty_qa:{i:04d}",
            "question": row["user_input"],
            "contexts": list(row.get("retrieved_contexts", row.get("contexts", []))),
            "answer": row["response"],
            "reference": row.get("reference", row.get("ground_truth", "")),
        })
    DATASET_SOURCES["amnesty_qa"] = "explodinggradients/amnesty_qa:english_v3/eval"
    return items[:LIMITS["amnesty_qa"]]

def load_hotpot():
    ds = load_dataset(
        "hotpotqa/hotpot_qa",
        "distractor",
        split=f"validation[:{LIMITS['hotpot_qa']}]",
    )
    items = []
    for i, row in enumerate(ds):
        contexts = [
            f"## {title}\n" + " ".join(sentences)
            for title, sentences in zip(
                row["context"]["title"],
                row["context"]["sentences"],
            )
        ]
        items.append({
            "item_id": f"hotpot_qa:{i:04d}",
            "question": row["question"],
            "contexts": contexts,
            "answer": row["answer"],
            "reference": row["answer"],
        })
    DATASET_SOURCES["hotpot_qa"] = "hotpotqa/hotpot_qa:distractor/validation"
    return items

def load_multihop():
    questions = load_dataset(
        "yixuantt/MultiHopRAG",
        "MultiHopRAG",
        split=f"train[:{LIMITS['multihop_rag']}]",
    )
    corpus = load_dataset(
        "yixuantt/MultiHopRAG",
        "corpus",
        split="train",
    )

    corpus_rows = list(corpus)
    by_url = {r.get("url"): r for r in corpus_rows if r.get("url")}
    by_id = {r.get("id"): r for r in corpus_rows if r.get("id")}

    def evidence_key(ev):
        return ev.get("url") or ev.get("id") if isinstance(ev, dict) else ev

    def render_doc(doc):
        return (
            f"## {doc.get('title', '')}\n"
            f"Source: {doc.get('source', '')}\n"
            f"Published: {doc.get('published_at', '')}\n"
            f"{doc.get('body', '')}"
        )

    items = []
    for i, row in enumerate(questions):
        evidence = []
        for ev in row["evidence_list"]:
            key = evidence_key(ev)
            doc = by_url.get(key) or by_id.get(key)
            if doc:
                evidence.append(doc)

        evidence_keys = {doc.get("url") or doc.get("id") for doc in evidence}
        candidates = [
            doc for doc in corpus_rows
            if (doc.get("url") or doc.get("id")) not in evidence_keys
        ]

        local_rng = np.random.default_rng(RNG_SEED + i)
        n_distractors = min(3, len(candidates))
        chosen = local_rng.choice(len(candidates), size=n_distractors, replace=False)
        distractors = [candidates[int(j)] for j in chosen]

        contexts = evidence + distractors
        local_rng.shuffle(contexts)

        items.append({
            "item_id": f"multihop_rag:{i:04d}",
            "question": row["query"],
            "contexts": [render_doc(doc) for doc in contexts],
            "answer": row["answer"],
            "reference": row["answer"],
        })

    DATASET_SOURCES["multihop_rag"] = (
        "yixuantt/MultiHopRAG:Hugging Face configs MultiHopRAG + corpus"
    )
    return items

DATASETS = {
    "amnesty_qa": load_amnesty(),
    "hotpot_qa": load_hotpot(),
    "multihop_rag": load_multihop(),
}

for name, items in DATASETS.items():
    print(f"{name:15s}: {len(items)} items")
    assert items
    assert all(item["question"] and item["answer"] and item["contexts"] for item in items)

print(json.dumps(DATASET_SOURCES, indent=2))

amnesty_qa     : 20 items
hotpot_qa      : 30 items
multihop_rag   : 30 items
{
  "amnesty_qa": "explodinggradients/amnesty_qa:english_v3/eval",
  "hotpot_qa": "hotpotqa/hotpot_qa:distractor/validation",
  "multihop_rag": "yixuantt/MultiHopRAG:Hugging Face configs MultiHopRAG + corpus"
}


## 4. Shared NiriZan Traces

Each benchmark item is passed through the real `Tracer` → `TraceCollector` → `CollectorExporter` path.

The trace structure is:

`PLANNING → RETRIEVAL → GENERATION`

The benchmark supplies the retrieval result and final answer. This isolates evaluator behavior instead of introducing a fourth application model whose generation quality would become another confounder.

In [4]:
class ListSink:
    def __init__(self):
        self.saved: list[Trace] = []

    async def save(self, trace: Trace) -> None:
        self.saved.append(trace)

async def build_traces(items):
    sink = ListSink()
    collector = TraceCollector(repository=sink)
    exporter = CollectorExporter(collector)
    tracer = Tracer(
        application_name="phase8-external-validation",
        exporter=exporter,
    )
    await collector.start()

    for item in items:
        async with tracer.start_span(
            "plan", SpanKind.PLANNING, input_payload=item["question"]
        ):
            async with tracer.start_span(
                "retrieve", SpanKind.RETRIEVAL, input_payload=item["question"]
            ) as retrieval:
                retrieval.output_payload = "\n\n".join(item["contexts"])

            async with tracer.start_span(
                "generate", SpanKind.GENERATION, input_payload=item["question"]
            ) as generation:
                generation.output_payload = item["answer"]

    await collector.stop()
    return sink.saved

ALL_ITEMS = [
    {"dataset": dataset, **item}
    for dataset, items in DATASETS.items()
    for item in items
]

TRACES = await build_traces(ALL_ITEMS)
ITEM_BY_TRACE_ID = {
    str(trace.trace_id): item
    for trace, item in zip(TRACES, ALL_ITEMS)
}

print("Traces:", len(TRACES))
assert len(TRACES) == len(ALL_ITEMS)
assert all(len(trace.spans) == 3 for trace in TRACES)

Traces: 80


## 5. NiriZan RAG Triad

NiriZan's `RAGTriadMetric` remains scorer-agnostic. The Sentence Transformers model is injected only inside this notebook, following notebook 02's established design.

The scorer is sanity-checked before benchmark scoring.

In [5]:
embedding_model = SentenceTransformer(
    "all-mpnet-base-v2",
    device="cuda" if __import__("torch").cuda.is_available() else "cpu",
)

def embedding_scorer(text_a: str, text_b: str) -> float:
    emb = embedding_model.encode(
        [text_a, text_b],
        convert_to_tensor=True,
        normalize_embeddings=True,
    )
    cosine = float(util.cos_sim(emb[0], emb[1]).item())
    return float(np.clip((cosine + 1.0) / 2.0, 0.0, 1.0))

relevant = embedding_scorer(
    "When was the Eiffel Tower completed?",
    "The Eiffel Tower was completed in 1889 for the World's Fair in Paris.",
)
irrelevant = embedding_scorer(
    "When was the Eiffel Tower completed?",
    "Bananas grow in tropical climates and contain potassium.",
)

print(f"relevant sanity score:   {relevant:.3f}")
print(f"irrelevant sanity score: {irrelevant:.3f}")
assert relevant - irrelevant > 0.20

rag_triad = RAGTriadMetric(scorer=embedding_scorer)
NIRIZAN_RESULTS = {}

for trace in TRACES:
    NIRIZAN_RESULTS[str(trace.trace_id)] = await rag_triad.evaluate(trace)

flat = [r for rs in NIRIZAN_RESULTS.values() for r in rs]
assert flat
assert all(0.0 <= r.score <= 1.0 for r in flat)

print("NiriZan MetricResults:", len(flat))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

relevant sanity score:   0.855
irrelevant sanity score: 0.523
NiriZan MetricResults: 240


# Section A — AmnestyQA

The complete 20-item AmnestyQA English evaluation split is scored by all four frameworks on the same normalized traces.

## 6A. External Framework Configuration

RAGAS uses its current Groq adapter. DeepEval uses `GPTModel` against Groq's OpenAI-compatible endpoint. Opik uses LiteLLM's `openai/` provider prefix against the same endpoint.

The external metric choices are fixed for all three datasets; no framework-specific metric is selected after seeing results.

In [6]:
# Cell 6 — External evaluator configuration
#
# RAGAS 0.4.3 + pinned Instructor stack.
# Groq is accessed through its OpenAI-compatible async endpoint.
#
# IMPORTANT:
# The VertexAI compatibility shim must be installed before importing RAGAS.

import sys
import types
from importlib.metadata import version as package_version


# ---------------------------------------------------------------------
# RAGAS / LangChain compatibility shim
# ---------------------------------------------------------------------

if "langchain_community.chat_models.vertexai" not in sys.modules:
    vertexai_module = types.ModuleType(
        "langchain_community.chat_models.vertexai"
    )

    vertexai_module.ChatVertexAI = type(
        "ChatVertexAI",
        (object,),
        {},
    )

    sys.modules[
        "langchain_community.chat_models.vertexai"
    ] = vertexai_module


# ---------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------

import instructor
import ragas

from openai import AsyncOpenAI

from ragas.llms import llm_factory
from ragas.embeddings import embedding_factory

from ragas.metrics.collections import (
    ContextPrecisionWithoutReference,
    Faithfulness,
    AnswerRelevancy,
)

from deepeval.metrics import (
    AnswerRelevancyMetric,
    ContextualRelevancyMetric,
    FaithfulnessMetric,
)

from deepeval.models import DeepEvalBaseLLM
from deepeval.test_case import LLMTestCase

from opik.evaluation.metrics import (
    AnswerRelevance,
    ContextPrecision,
    Hallucination,
)


# ---------------------------------------------------------------------
# Canonical Groq model identifiers
# ---------------------------------------------------------------------

# GROQ_OPENAI_MODEL is the literal model ID Groq's REST API expects.
# The "openai/" segment here is part of Groq's own model slug, not a
# routing prefix, so this is what AsyncOpenAI / Instructor (RAGAS,
# DeepEval) must send.
GROQ_OPENAI_MODEL = JUDGE_MODEL

if not GROQ_OPENAI_MODEL.startswith("openai/"):
    GROQ_OPENAI_MODEL = f"openai/{GROQ_OPENAI_MODEL}"

# OPIK_LITELLM_MODEL is for Opik, which resolves models through litellm
# rather than calling Groq's endpoint directly. litellm treats a leading
# "openai/" as "route to OpenAI's own API," stripping it and losing the
# real model ID, which is why Opik was requesting a bare "gpt-oss-20b"
# from Groq and getting a 404. litellm's own registry confirms the
# correct string is "groq/" + the full Groq model ID, i.e.
# "groq/openai/gpt-oss-20b", so build it explicitly for Opik.
OPIK_LITELLM_MODEL = f"groq/{GROQ_OPENAI_MODEL}"


# ---------------------------------------------------------------------
# RAGAS client
# ---------------------------------------------------------------------

ragas_client = AsyncOpenAI(
    api_key=GROQ_API_KEY,
    base_url=JUDGE_BASE_URL,
)

ragas_llm = llm_factory(
    GROQ_OPENAI_MODEL,
    provider="openai",
    client=ragas_client,
    adapter="instructor",
    temperature=0.0,
    max_tokens=4096,
    reasoning_effort="low",
)


# ---------------------------------------------------------------------
# RAGAS embeddings
# ---------------------------------------------------------------------

ragas_embeddings = embedding_factory(
    "huggingface",
    model="sentence-transformers/all-MiniLM-L6-v2",
)


# ---------------------------------------------------------------------
# RAGAS metrics
# ---------------------------------------------------------------------

RAGAS_METRICS = {
    "context_relevance": ContextPrecisionWithoutReference(
        llm=ragas_llm,
    ),
    "groundedness": Faithfulness(
        llm=ragas_llm,
    ),
    "answer_relevance": AnswerRelevancy(
        llm=ragas_llm,
        embeddings=ragas_embeddings,
    ),
}


# ---------------------------------------------------------------------
# DeepEval
# ---------------------------------------------------------------------

class GroqDeepEvalModel(DeepEvalBaseLLM):

    def __init__(self):
        self.client = AsyncOpenAI(
            api_key=GROQ_API_KEY,
            base_url=JUDGE_BASE_URL,
        )

    def load_model(self):
        return self.client

    def generate(self, prompt: str, schema=None):
        import asyncio
        import concurrent.futures

        async def _generate():
            response = await self.client.chat.completions.create(
                model=GROQ_OPENAI_MODEL,
                messages=[
                    {
                        "role": "user",
                        "content": prompt,
                    }
                ],
                temperature=0.0,
                max_tokens=4096,
                reasoning_effort="low",
            )

            return response.choices[0].message.content

        try:
            asyncio.get_running_loop()
        except RuntimeError:
            return asyncio.run(_generate())

        with concurrent.futures.ThreadPoolExecutor(
            max_workers=1
        ) as executor:
            return executor.submit(
                asyncio.run,
                _generate(),
            ).result()

    async def a_generate(self, prompt: str, schema=None):
        response = await self.client.chat.completions.create(
            model=GROQ_OPENAI_MODEL,
            messages=[
                {
                    "role": "user",
                    "content": prompt,
                }
            ],
            temperature=0.0,
            max_tokens=4096,
            reasoning_effort="low",
        )

        return response.choices[0].message.content

    def get_model_name(self):
        return f"Groq/{GROQ_OPENAI_MODEL}"


deepeval_model = GroqDeepEvalModel()

DEEPEVAL_METRICS = {
    "context_relevance": ContextualRelevancyMetric(
        model=deepeval_model,
        threshold=0.0,
        include_reason=False,
    ),
    "groundedness": FaithfulnessMetric(
        model=deepeval_model,
        threshold=0.0,
        include_reason=False,
    ),
    "answer_relevance": AnswerRelevancyMetric(
        model=deepeval_model,
        threshold=0.0,
        include_reason=False,
    ),
}


# ---------------------------------------------------------------------
# Opik
# ---------------------------------------------------------------------

OPIK_METRICS = {
    "context_relevance": ContextPrecision(
        model=OPIK_LITELLM_MODEL,
    ),
    "groundedness": Hallucination(
        model=OPIK_LITELLM_MODEL,
    ),
    "answer_relevance": AnswerRelevance(
        model=OPIK_LITELLM_MODEL,
    ),
}


# ---------------------------------------------------------------------
# Version / configuration checks
# ---------------------------------------------------------------------

RAGAS_VERSION = package_version("ragas")
INSTRUCTOR_VERSION = package_version("instructor")
OPENAI_VERSION = package_version("openai")

print("RAGAS version:", RAGAS_VERSION)
print("Instructor version:", INSTRUCTOR_VERSION)
print("OpenAI version:", OPENAI_VERSION)

print("RAGAS client:", type(ragas_client).__name__)
print("RAGAS async:", ragas_llm.is_async)
print("RAGAS adapter: instructor")
print("Judge model (Groq REST):", GROQ_OPENAI_MODEL)
print("Judge model (litellm/Opik):", OPIK_LITELLM_MODEL)
print("Judge endpoint:", JUDGE_BASE_URL)

assert RAGAS_VERSION == "0.4.3"
assert isinstance(ragas_client, AsyncOpenAI)
assert ragas_llm.is_async is True
assert str(ragas_client.base_url).rstrip("/") == JUDGE_BASE_URL.rstrip("/")

print("External evaluator configuration ready.")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

RAGAS version: 0.4.3
Instructor version: 1.14.3
OpenAI version: 2.45.0
RAGAS client: AsyncOpenAI
RAGAS async: True
RAGAS adapter: instructor
Judge model (Groq REST): openai/gpt-oss-20b
Judge model (litellm/Opik): groq/openai/gpt-oss-20b
Judge endpoint: https://api.groq.com/openai/v1
External evaluator configuration ready.


In [7]:
import asyncio
import inspect
import random
import re


def init_cache(path):
    conn = sqlite3.connect(path)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS scores (
            item_id TEXT NOT NULL,
            dataset TEXT NOT NULL,
            framework TEXT NOT NULL,
            metric TEXT NOT NULL,
            score REAL NOT NULL,
            latency_ms REAL NOT NULL,
            judge_model TEXT NOT NULL,
            computed_at TEXT NOT NULL,
            PRIMARY KEY (item_id, framework, metric, judge_model)
        )
    """)
    conn.commit()
    return conn


CACHE = init_cache(CACHE_PATH)


def has_score(item_id, framework, metric):
    return CACHE.execute(
        """SELECT 1 FROM scores
           WHERE item_id=? AND framework=? AND metric=? AND judge_model=?""",
        (item_id, framework, metric, GROQ_MODEL),
    ).fetchone() is not None


def cache_score(item, framework, metric, score, latency_ms):
    CACHE.execute(
        """INSERT OR REPLACE INTO scores
           VALUES (?, ?, ?, ?, ?, ?, ?, ?)""",
        (
            item["item_id"],
            item["dataset"],
            framework,
            metric,
            float(score),
            float(latency_ms),
            GROQ_MODEL,
            datetime.now(timezone.utc).isoformat(),
        ),
    )
    CACHE.commit()


# ---------------------------------------------------------------------------
# Rate-limit retry
# ---------------------------------------------------------------------------
#
# Groq's free-tier TPM budget (e.g. 8000 tokens/minute for gpt-oss-20b) is
# small relative to the number of judge calls per item: RAGAS, DeepEval,
# and Opik each fire up to 3 sequential calls, so a single item can exhaust
# the whole per-minute budget. Opik's own internal tenacity retry gives up
# after a short budget and re-raises rather than pausing indefinitely, so
# this wraps each framework's scoring call at the notebook level and
# retries on rate-limit errors, honoring Groq's own "try again in Xs" hint
# when present.

RATE_LIMIT_MAX_RETRIES = 8
RATE_LIMIT_BASE_DELAY = 5.0
RATE_LIMIT_MAX_DELAY = 90.0

_RETRY_AFTER_RE = re.compile(r"try again in ([\d.]+)\s*s", re.IGNORECASE)


def _error_text(exc) -> str:
    text = str(exc)
    cause = getattr(exc, "__cause__", None)
    if cause is not None:
        text += " " + str(cause)
    return text


def _looks_like_rate_limit(exc) -> bool:
    text = _error_text(exc).lower()
    return (
        "rate_limit" in text
        or "rate limit" in text
        or "429" in text
        or "ratelimiterror" in exc.__class__.__name__.lower()
    )


def _retry_after_seconds(exc, default: float) -> float:
    match = _RETRY_AFTER_RE.search(_error_text(exc))
    if match:
        return float(match.group(1)) + 1.0
    return default


async def call_with_rate_limit_retry(fn):
    """
    Call fn() (sync, or returning an awaitable) and retry on rate-limit
    errors from Groq / litellm / Opik with backoff.
    """
    delay = RATE_LIMIT_BASE_DELAY

    for attempt in range(1, RATE_LIMIT_MAX_RETRIES + 1):
        try:
            result = fn()
            if inspect.isawaitable(result):
                result = await result
            return result
        except Exception as exc:
            if not _looks_like_rate_limit(exc) or attempt == RATE_LIMIT_MAX_RETRIES:
                raise

            wait_s = min(
                _retry_after_seconds(exc, default=delay),
                RATE_LIMIT_MAX_DELAY,
            )
            wait_s += random.uniform(0, 1.0)

            print(
                f"\n[rate limit] attempt {attempt}/{RATE_LIMIT_MAX_RETRIES}, "
                f"waiting {wait_s:.1f}s..."
            )

            await asyncio.sleep(wait_s)
            delay = min(delay * 2, RATE_LIMIT_MAX_DELAY)


# ---------------------------------------------------------------------------
# RAGAS
# ---------------------------------------------------------------------------
#
# The selected RAGAS metrics have different input requirements:
#
# ContextPrecisionWithoutReference:
#     user_input + response + retrieved_contexts
#
# Faithfulness:
#     user_input + response + retrieved_contexts
#
# AnswerRelevancy:
#     user_input + response
#
# None of these metrics accepts a `reference` argument.
# The reference answer remains available in `item["reference"]` for the
# reference-aware DeepEval and Opik evaluations below.

async def score_ragas_item(item):
    values = {}

    for dim, metric in RAGAS_METRICS.items():

        if has_score(item["item_id"], "ragas", dim):
            continue

        started = time.perf_counter()

        if dim == "context_relevance":
            result = await metric.ascore(
                user_input=item["question"],
                response=item["answer"],
                retrieved_contexts=item["contexts"],
            )

        elif dim == "groundedness":
            result = await metric.ascore(
                user_input=item["question"],
                response=item["answer"],
                retrieved_contexts=item["contexts"],
            )

        elif dim == "answer_relevance":
            result = await metric.ascore(
                user_input=item["question"],
                response=item["answer"],
            )

        else:
            raise ValueError(
                f"Unsupported RAGAS dimension: {dim}"
            )

        score = float(result.value)

        if not 0.0 <= score <= 1.0:
            raise ValueError(
                f"RAGAS returned an invalid score for "
                f"{dim}: {score}"
            )

        values[dim] = (
            score,
            (time.perf_counter() - started) * 1000.0,
        )

    return values


# ---------------------------------------------------------------------------
# DeepEval
# ---------------------------------------------------------------------------

def score_deepeval_item(item):
    case = LLMTestCase(
        input=item["question"],
        actual_output=item["answer"],
        retrieval_context=item["contexts"],
        expected_output=item["reference"],
    )

    values = {}

    for dim, metric in DEEPEVAL_METRICS.items():

        if has_score(item["item_id"], "deepeval", dim):
            continue

        started = time.perf_counter()

        metric.measure(case)

        score = float(metric.score)

        if not 0.0 <= score <= 1.0:
            raise ValueError(
                f"DeepEval returned an invalid score for "
                f"{dim}: {score}"
            )

        values[dim] = (
            score,
            (time.perf_counter() - started) * 1000.0,
        )

    return values


# ---------------------------------------------------------------------------
# Opik
# ---------------------------------------------------------------------------

def score_opik_item(item):
    values = {}

    for dim, metric in OPIK_METRICS.items():

        if has_score(item["item_id"], "opik", dim):
            continue

        started = time.perf_counter()

        result = metric.score(
            input=item["question"],
            output=item["answer"],
            context=item["contexts"],
            expected_output=item["reference"],
        )

        score = float(result.value)

        # Opik's Hallucination metric is:
        #     1.0 = hallucination detected
        #     0.0 = no hallucination
        #
        # NiriZan's evaluation convention is higher-is-better, so invert it
        # to obtain a groundedness-style score.

        if dim == "groundedness":
            score = 1.0 - score

        if not 0.0 <= score <= 1.0:
            raise ValueError(
                f"Opik returned an invalid score for "
                f"{dim}: {score}"
            )

        values[dim] = (
            score,
            (time.perf_counter() - started) * 1000.0,
        )

    return values


# ---------------------------------------------------------------------------
# NiriZan cache population
# ---------------------------------------------------------------------------

def cache_nirizan(dataset_name):
    for trace_id, item in ITEM_BY_TRACE_ID.items():

        if item["dataset"] != dataset_name:
            continue

        for result in NIRIZAN_RESULTS[trace_id]:

            if not has_score(
                item["item_id"],
                "nirizan",
                result.metric_name,
            ):
                cache_score(
                    item,
                    "nirizan",
                    result.metric_name,
                    result.score,
                    0.0,
                )


# ---------------------------------------------------------------------------
# Dataset evaluation
# ---------------------------------------------------------------------------

async def evaluate_external_dataset(dataset_name, items):
    cache_nirizan(dataset_name)

    # `items` may be the raw DATASETS[...] list, whose entries do not
    # carry a "dataset" key (only ALL_ITEMS / ITEM_BY_TRACE_ID do, per
    # cell 8). cache_score() requires item["dataset"], so stamp it here
    # rather than assuming the caller already tagged each item.
    for item in items:
        item.setdefault("dataset", dataset_name)

    for item in items:

        # ---------------------------------------------------------------
        # RAGAS
        # ---------------------------------------------------------------

        ragas_values = await call_with_rate_limit_retry(
            lambda item=item: score_ragas_item(item)
        )

        for dim, (score, latency) in ragas_values.items():
            cache_score(
                item,
                "ragas",
                dim,
                score,
                latency,
            )

        # ---------------------------------------------------------------
        # DeepEval
        # ---------------------------------------------------------------

        deepeval_values = await call_with_rate_limit_retry(
            lambda item=item: score_deepeval_item(item)
        )

        for dim, (score, latency) in deepeval_values.items():
            cache_score(
                item,
                "deepeval",
                dim,
                score,
                latency,
            )

        # ---------------------------------------------------------------
        # Opik
        # ---------------------------------------------------------------

        opik_values = await call_with_rate_limit_retry(
            lambda item=item: score_opik_item(item)
        )

        for dim, (score, latency) in opik_values.items():
            cache_score(
                item,
                "opik",
                dim,
                score,
                latency,
            )

        print(".", end="", flush=True)

    print()


print("Helpers ready; live judge calls begin in Section A.")

Helpers ready; live judge calls begin in Section A.


## 7A. AmnestyQA — Execute All Four Frameworks

This is the first live external evaluation. Successful results are persisted immediately.

In [8]:
await evaluate_external_dataset("amnesty_qa", DATASETS["amnesty_qa"])
print("AmnestyQA complete.")

..

Output()


[rate limit] attempt 1/8, waiting 5.5s...


Output()


[rate limit] attempt 2/8, waiting 10.5s...


...


[rate limit] attempt 6/8, waiting 6.6s...


OPIK: Failed to call LLM provider, reason: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01m02kwf6fejybe0wtc015z06y` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 7390, Requested 1342. Please try again in 5.49s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}




[rate limit] attempt 7/8, waiting 7.1s...


OPIK: Failed to call LLM provider, reason: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01m02kwf6fejybe0wtc015z06y` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 6200, Requested 2100. Please try again in 2.25s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}



BaseLLMError: LLM infrastructure error: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01m02kwf6fejybe0wtc015z06y` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 6200, Requested 2100. Please try again in 2.25s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}


# Section B — HotpotQA

HotpotQA extends the evaluation to natural multi-hop questions with distractor contexts. The same framework configuration and judge model are reused without modification.

In [ ]:
await evaluate_external_dataset("hotpot_qa", DATASETS["hotpot_qa"])
print("HotpotQA complete.")

# Section C — MultiHop-RAG

MultiHop-RAG is the dedicated multi-document RAG benchmark in this experiment. The default 30-item slice is intentionally bounded for Groq free-tier reproducibility; the loader can be scaled after quota has been confirmed.

In [ ]:
await evaluate_external_dataset("multihop_rag", DATASETS["multihop_rag"])
print("MultiHop-RAG complete.")

## 8. Score Integrity

Missing scores are not converted to zero. This prevents API failures or framework parsing failures from becoming artificial disagreement.

In [ ]:
scores_df = pd.read_sql_query("SELECT * FROM scores", CACHE)

assert not scores_df.empty
assert scores_df["score"].between(0, 1).all()

expected_items = sum(LIMITS.values())
for framework in ["nirizan", "ragas", "deepeval", "opik"]:
    count = scores_df.loc[
        scores_df.framework == framework, "item_id"
    ].nunique()
    print(f"{framework:10s}: {count}/{expected_items}")
    assert count == expected_items, f"{framework} did not score every item"

print(scores_df.groupby(["dataset", "framework"])["metric"].count().unstack(fill_value=0))

## 9. Agreement Analysis by RAG-Triad Dimension

The primary external-validation statistic is **Spearman's rho**, computed between NiriZan and each external framework for the same benchmark items and the same quality dimension.

Because the native scales differ, a secondary normalized MAE is calculated after min-max normalization within each paired comparison.

A high rho means that two evaluators rank items similarly. It does not mean their raw scores are interchangeable.

In [ ]:
FRAMEWORK_METRICS = {
    "ragas": {
        "context_relevance": "context_relevance",
        "groundedness": "groundedness",
        "answer_relevance": "answer_relevance",
    },
    "deepeval": {
        "context_relevance": "context_relevance",
        "groundedness": "groundedness",
        "answer_relevance": "answer_relevance",
    },
    "opik": {
        "context_relevance": "context_relevance",
        "groundedness": "groundedness",
        "answer_relevance": "answer_relevance",
    },
}

def minmax(values):
    x = np.asarray(values, dtype=float)
    lo, hi = x.min(), x.max()
    return np.zeros_like(x) if hi - lo < 1e-12 else (x - lo) / (hi - lo)

agreement_rows = []

for dataset_name in DATASETS:
    for framework in ["ragas", "deepeval", "opik"]:
        for dimension in ["context_relevance", "groundedness", "answer_relevance"]:
            nz = scores_df[
                (scores_df.dataset == dataset_name)
                & (scores_df.framework == "nirizan")
                & (scores_df.metric == dimension)
            ][["item_id", "score"]].rename(columns={"score": "nirizan"})

            ext = scores_df[
                (scores_df.dataset == dataset_name)
                & (scores_df.framework == framework)
                & (scores_df.metric == FRAMEWORK_METRICS[framework][dimension])
            ][["item_id", "score"]].rename(columns={"score": "external"})

            pair = nz.merge(ext, on="item_id")
            assert len(pair) == LIMITS[dataset_name]

            a = minmax(pair["nirizan"])
            b = minmax(pair["external"])
            rho, p = scipy_stats.spearmanr(a, b)

            agreement_rows.append({
                "dataset": dataset_name,
                "framework": framework,
                "dimension": dimension,
                "n": len(pair),
                "spearman_rho": float(rho),
                "spearman_p": float(p),
                "normalized_mae": float(np.mean(np.abs(a - b))),
            })

agreement_df = pd.DataFrame(agreement_rows)
display(
    agreement_df.sort_values(["dataset", "dimension", "framework"])
)

## 10. Cross-Dataset Agreement Profile

This view makes the robustness question explicit: does agreement with NiriZan persist when moving from a small curated RAG dataset to distractor-heavy multi-hop QA and then to a dedicated multi-document RAG benchmark?

In [ ]:
agreement_pivot = agreement_df.pivot_table(
    index=["dataset", "dimension"],
    columns="framework",
    values="spearman_rho",
)
display(agreement_pivot)

## 11. Operational Latency

External latency is measured around the actual metric invocation and includes framework overhead plus Groq network/inference time. NiriZan's embedding path is reported separately because it does not call the Groq judge.

The notebook does not convert latency to a dollar estimate because this experiment intentionally uses Groq's free tier.

In [ ]:
latency_df = (
    scores_df[scores_df.framework != "nirizan"]
    .groupby(["dataset", "framework", "metric"], as_index=False)
    .agg(
        n=("latency_ms", "count"),
        p50_ms=("latency_ms", "median"),
        p95_ms=("latency_ms", lambda x: np.percentile(x, 95)),
        mean_ms=("latency_ms", "mean"),
    )
)
display(latency_df)

## 12. Statistical Scaling: False Alarms and Power

No additional Groq calls are made here.

The experiment uses the observed NiriZan groundedness distribution from the real benchmark pool as the center of an A/A simulation and a 10% relative degradation simulation. This extends notebook 07's isolated operating points into a sample-size curve.

The result answers a narrow engineering question: how much evidence does NiriZan's statistical gate need to detect a practically meaningful degradation while controlling false alarms?

In [ ]:
nz_groundedness = scores_df[
    (scores_df.framework == "nirizan")
    & (scores_df.metric == "groundedness")
]["score"].to_numpy()

observed_mean = float(nz_groundedness.mean())
observed_std = float(nz_groundedness.std(ddof=1))

SAMPLE_SIZES = [10, 20, 30, 50, 75, 100, 150, 200]
N_TRIALS = 200
DROP = 0.10

comparator = BaselineComparator()

def simulated_pair(n, degraded, seed):
    rng = np.random.default_rng(seed)
    baseline = {
        "groundedness": np.clip(
            rng.normal(observed_mean, observed_std, n), 0, 1
        )
    }
    mean = observed_mean * (1 - DROP) if degraded else observed_mean
    candidate = {
        "groundedness": np.clip(
            rng.normal(mean, observed_std, n), 0, 1
        )
    }
    return baseline, candidate

scaling_rows = []

for n in SAMPLE_SIZES:
    false_alarms = 0
    detections = 0

    for trial in range(N_TRIALS):
        baseline, candidate = simulated_pair(
            n, False, n * 10_000 + trial
        )
        verdict = comparator.compare(
            candidate_scores=candidate,
            baseline_scores=baseline,
            baseline_id=uuid4(),
            run_id=uuid4(),
        )[0]
        false_alarms += verdict.severity == RegressionSeverity.BLOCKING

        baseline, candidate = simulated_pair(
            n, True, n * 20_000 + trial
        )
        verdict = comparator.compare(
            candidate_scores=candidate,
            baseline_scores=baseline,
            baseline_id=uuid4(),
            run_id=uuid4(),
        )[0]
        detections += verdict.severity != RegressionSeverity.NONE

    scaling_rows.append({
        "n": n,
        "false_alarm_rate": false_alarms / N_TRIALS,
        "power": detections / N_TRIALS,
    })

scaling_df = pd.DataFrame(scaling_rows)
display(scaling_df)

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5))
ax.plot(
    scaling_df["n"],
    scaling_df["false_alarm_rate"],
    "o-",
    label="false alarm rate",
)
ax.plot(
    scaling_df["n"],
    scaling_df["power"],
    "s-",
    label="power for 10% drop",
)
ax.axhline(0.05, linestyle="--", label="5% false-alarm target")
ax.axhline(0.95, linestyle="--", label="95% power target")
ax.set_xlabel("Sample size (N)")
ax.set_ylabel("Rate")
ax.set_title("NiriZan statistical gate: false alarms and power")
ax.legend()
fig.tight_layout()
fig.savefig(RESULTS_DIR / "nirizan_power_curve.pdf", bbox_inches="tight")
plt.show()

recommended_n = next(
    (int(row.n) for row in scaling_df.itertuples()
     if row.false_alarm_rate <= 0.05 and row.power >= 0.95),
    None,
)
print("First tested N meeting both targets:", recommended_n)

## 13. Benchmark Score Summary

This table is descriptive. It is not a framework leaderboard because the metric implementations are not identical.

In [ ]:
summary_df = (
    scores_df
    .groupby(["dataset", "framework", "metric"], as_index=False)
    .agg(
        n=("score", "count"),
        mean=("score", "mean"),
        std=("score", "std"),
        median=("score", "median"),
    )
)
display(summary_df.sort_values(["dataset", "framework", "metric"]))

## 14. Reproducibility Manifest

The manifest records the exact judge model, benchmark sources, sample limits, metric mapping, random seed, and installed package versions.

LLM API reproducibility means preserving configuration and cached results; it does not guarantee that a future provider invocation will be bit-identical.

In [ ]:
import importlib.metadata as md

def version_of(name):
    try:
        return md.version(name)
    except md.PackageNotFoundError:
        return None

manifest = {
    "notebook": "08_external_validation_and_scale.ipynb",
    "nirizan_version": NIRIZAN_VERSION,
    "python_version": sys.version,
    "judge_provider": "Groq",
    "judge_model": GROQ_MODEL,
    "judge_base_url": JUDGE_BASE_URL,
    "temperature": 0.0,
    "dataset_limits": LIMITS,
    "dataset_sources": DATASET_SOURCES,
    "random_seed": RNG_SEED,
    "metric_mapping": {
        "context_relevance": {
            "nirizan": "context_relevance",
            "ragas": "ContextPrecisionWithoutReference",
            "deepeval": "ContextualRelevancyMetric",
            "opik": "ContextPrecision",
        },
        "groundedness": {
            "nirizan": "groundedness",
            "ragas": "Faithfulness",
            "deepeval": "FaithfulnessMetric",
            "opik": "1 - Hallucination",
        },
        "answer_relevance": {
            "nirizan": "answer_relevance",
            "ragas": "AnswerRelevancy",
            "deepeval": "AnswerRelevancyMetric",
            "opik": "AnswerRelevance",
        },
    },
    "package_versions": {
        name: version_of(name)
        for name in [
            "nirizan", "ragas", "deepeval", "opik",
            "groq", "litellm", "sentence-transformers",
        ]
    },
    "cache_path": str(CACHE_PATH),
}

(RESULTS_DIR / "reproducibility_manifest.json").write_text(
    json.dumps(manifest, indent=2)
)
print(json.dumps(manifest, indent=2))

## 15. Threats to Validity

**Shared judge dependence.** RAGAS, DeepEval, and Opik use the same Groq judge. This isolates framework differences from provider/model differences, but it does not establish human-level validity.

**Metric non-equivalence.** The mapped metrics are conceptually aligned, not mathematically identical. Opik's `ContextPrecision` is reference-oriented; DeepEval's `ContextualRelevancyMetric` is referenceless.

**Public benchmark contamination.** These datasets are public and may have been encountered by judge or embedding models during training.

**Bounded sample.** The live experiment uses 80 items for quota discipline. MultiHop-RAG itself contains 2,556 queries; the default slice is not a claim that 30 items fully represents the benchmark.

**Dataset dependence.** AmnestyQA, HotpotQA, and MultiHop-RAG exercise different retrieval and reasoning behaviors. Results should be reported separately before any pooled conclusion.

**Evaluator-only design.** The benchmark supplies the application answer. This is intentional: the experiment measures whether different evaluation frameworks produce similar judgments on identical RAG traces, rather than comparing application generation systems.

**Groq free-tier variability.** Rate limits and provider-side nondeterminism can affect completion timing and exact judge outputs. The notebook therefore caches raw results and records the judge configuration.

## 16. Production Extraction Map

This experiment validates NiriZan; it does not define a new production contract.

| Notebook component | Potential recurring target | Status |
|---|---|---|
| Dataset adapters | `benchmarks/datasets/` | Evaluation harness only |
| Framework adapters | `benchmarks/external_comparison.py` | Evaluation harness only |
| SQLite score cache | `benchmarks/cache.py` | Evaluation harness only |
| Agreement analysis | `benchmarks/agreement.py` | Evaluation harness only |
| Power analysis | `benchmarks/power_analysis.py` | Evaluation harness only |
| Reproducibility manifest | `benchmarks/reproducibility.py` | Evaluation harness only |
| `src/nirizan/` | Existing package | **No changes proposed** |

No new `contracts.md` model is introduced by this notebook.

## Conclusion

This notebook supplies an external-validation layer for NiriZan across three public RAG benchmarks and three established external evaluation frameworks.

The primary research outputs are:

1. dimension-level Spearman agreement between NiriZan and each external framework;
2. cross-dataset agreement profiles for AmnestyQA, HotpotQA, and MultiHop-RAG;
3. measured external-framework latency;
4. empirical NiriZan false-alarm/power curves without additional LLM calls; and
5. a reproducibility manifest plus persistent raw-score cache.

The defensible interpretation is **comparative validation**, not a universal framework ranking. The strongest evidence comes from agreement by RAG-Triad dimension, persistence across benchmark types, and transparent reporting of metric and judge limitations.